## ouvrir_8347_et_trier_renvois_STOCK

**Fichier(s) source :** `./data/bofip_stock_live_20260521.tgz` (stock complet du 21.05.2026, pas un flux) et `./data/inventaire_bofip_stock_live_20260521.xlsx` (inventaire produit par `profilage_bofip_consolide.ipynb`)

**Fichier(s) de sortie :** `./data/extrait_8347-PGP/8347-PGP.html` (extrait HTML facultatif, Étape 6)

**Description :** Ouvre un document du stock BOFiP (cas 8347-PGP), lit ses renvois (dc:relation), les trie par sorte (Contenu, Actualité, Fichier), puis vérifie chaque cible contre l'inventaire pour distinguer les renvois résolus des renvois orphelins.

## Etape 1. Indiquer les chemins (archive de STOCK et inventaire)

In [1]:
import tarfile, os, glob, re
import html as _html

# Archive du STOCK complet (.tgz). Ne pas mettre un fichier de flux.
TGZ_PATH = r"./data/bofip_stock_live_20260521.tgz"
# Inventaire (.xlsx)
INVENTAIRE_PATH = r"./data/inventaire_bofip_stock_live_20260521.xlsx"

BASE = r"./data"

def _chercher(motifs, racine):
    res=[]
    if os.path.isdir(racine):
        for m in motifs:
            res += glob.glob(os.path.join(racine, '**', m), recursive=True)
    return res

def _choisir_stock(candidats):
    # Preferer une archive 'stock', eviter 'flux'.
    stock = [c for c in candidats if 'stock' in os.path.basename(c).lower()]
    if stock: return stock[0]
    non_flux = [c for c in candidats if 'flux' not in os.path.basename(c).lower()]
    if non_flux: return non_flux[0]
    return candidats[0] if candidats else None

# Si le chemin par defaut n'existe pas, rechercher automatiquement une archive de stock.
if not os.path.exists(TGZ_PATH):
    cand = _chercher(['*.tgz','*.tar.gz'], BASE)
    choix = _choisir_stock(cand)
    if choix: TGZ_PATH = choix

if not os.path.exists(INVENTAIRE_PATH):
    inv = _chercher(['inventaire*stock*.xlsx','inventaire*.xlsx'], BASE)
    if inv: INVENTAIRE_PATH = inv[0]

print('Archive    :', TGZ_PATH if os.path.exists(TGZ_PATH) else 'INTROUVABLE (renseigner TGZ_PATH)')
print('Inventaire :', INVENTAIRE_PATH if os.path.exists(INVENTAIRE_PATH) else 'INTROUVABLE (renseigner INVENTAIRE_PATH)')

# Avertissement si l'archive ressemble a un flux
if os.path.exists(TGZ_PATH) and 'flux' in os.path.basename(TGZ_PATH).lower():
    print()
    print('ATTENTION : l\'archive choisie semble etre un FLUX, pas le stock complet.')
    print('Renseigner TGZ_PATH avec le fichier de stock (nom contenant "stock").')

Archive    : ./data/bofip_stock_live_20260521.tgz
Inventaire : ./data/inventaire_bofip_stock_live_20260521.xlsx


## Etape 2. Lire la liste des fichiers de l'archive

Le stock complet contient des milliers de fichiers. Si ce nombre est tres petit (par exemple une dizaine), l'archive est probablement un flux : revenir a l'Etape 1.

In [2]:
NOMS_MEMBRES = []
if os.path.exists(TGZ_PATH):
    with tarfile.open(TGZ_PATH, 'r:gz') as tar:
        NOMS_MEMBRES = [m.name for m in tar.getmembers() if m.isfile()]
    print('Nombre de fichiers dans l\'archive :', len(NOMS_MEMBRES))
    if len(NOMS_MEMBRES) < 100:
        print('Ce nombre est tres petit : il s\'agit probablement d\'un flux, pas du stock complet.')
else:
    print('Archive introuvable : revenir a l\'Etape 1.')

Nombre de fichiers dans l'archive : 14993


## Etape 3. Retrouver les fichiers du document

La variable `IDENTIFIANT` est pre-reglee sur 8347-PGP (modifiable).

In [3]:
IDENTIFIANT = '8347-PGP'

def appartient(nom):
    for s in re.split(r'[\\/]', nom):
        if s == IDENTIFIANT or s.split('.')[0] == IDENTIFIANT:
            return True
    return False

fichiers_doc = [n for n in NOMS_MEMBRES if appartient(n)]
print('Fichiers rattaches a', IDENTIFIANT, ':', len(fichiers_doc))
for n in fichiers_doc:
    print('   ', n)
if not fichiers_doc and NOMS_MEMBRES:
    apercu = [n for n in NOMS_MEMBRES if IDENTIFIANT.split('-')[0] in n]
    if apercu:
        print()
        print('Noms approchants trouves :')
        for n in apercu[:10]:
            print('   ', n)

Fichiers rattaches a 8347-PGP : 2
    BOFiP/documents/Contenu/Commentaire/CAD/8347-PGP/2021-05-12/document.xml
    BOFiP/documents/Contenu/Commentaire/CAD/8347-PGP/2021-05-12/data.html


## Etape 4. Lire les renvois et les trier par sorte

Chaque renvoi s'ecrit `Sorte:identifiant`, par exemple `Contenu:162-PGP`, `Actualite:13498-PGP` ou `Fichier:...`. Cette etape extrait tous les renvois et les regroupe par sorte.

In [4]:
renvois = []
if fichiers_doc and os.path.exists(TGZ_PATH):
    xmls = [n for n in fichiers_doc if n.lower().endswith('.xml')]
    if xmls:
        with tarfile.open(TGZ_PATH, 'r:gz') as tar:
            texte = tar.extractfile(xmls[0]).read().decode('utf-8', errors='replace')
        renvois = re.findall(r'<dc:relation[^>]*>([^<]+)</dc:relation>', texte)

groupes = {}
for r in renvois:
    r = r.strip()
    sorte = r.split(':',1)[0] if ':' in r else 'Autre'
    cible = r.split(':',1)[1] if ':' in r else r
    cible = cible.split('#',1)[0]
    groupes.setdefault(sorte, []).append(cible)

print('Total des renvois :', len(renvois))
for sorte, lst in groupes.items():
    print()
    print(f'--- {sorte} ({len(lst)}) ---')
    for c in lst:
        print('   ', c)

Total des renvois : 15

--- Actualite (1) ---
    12500-PGP

--- Fichier (6) ---
    13093-PGP
    13694-PGP
    13695-PGP
    13705-PGP
    13706-PGP
    14122-PGP

--- Contenu (8) ---
    13690-PGP
    13691-PGP
    5176-PGP
    5251-PGP
    5261-PGP
    5266-PGP
    5312-PGP
    8346-PGP


## Etape 5. Verifier chaque cible : resolue ou orpheline

Charge la liste des identifiants du stock (depuis l'inventaire) et verifie, pour chaque renvoi, si sa cible est presente. Les actualites et les fichiers ne figurent pas parmi les documents de l'inventaire : ils ressortent donc comme hors stock.

Remarque : si l'inventaire est ouvert dans Excel, sa lecture peut echouer (acces refuse). Le fermer dans Excel puis reexecuter cette cellule.

In [5]:
identifiants_stock = set()
if os.path.exists(INVENTAIRE_PATH):
    try:
        import pandas as pd
        col = pd.read_excel(INVENTAIRE_PATH, usecols=['identifiant'])['identifiant'].astype(str)
        identifiants_stock = set(col.tolist())
    except Exception as e:
        try:
            import openpyxl
            wb = openpyxl.load_workbook(INVENTAIRE_PATH, data_only=True, read_only=True)
            ws = wb.active
            entetes = [c.value for c in next(ws.iter_rows(min_row=1, max_row=1))]
            j = entetes.index('identifiant')
            for row in ws.iter_rows(min_row=2, values_only=True):
                if row[j] is not None:
                    identifiants_stock.add(str(row[j]))
        except Exception as e2:
            print('Lecture de l\'inventaire impossible (le fermer dans Excel puis reexecuter).')
            print('Detail :', e2)
    print('Identifiants connus dans le stock :', len(identifiants_stock))
else:
    print('Inventaire introuvable : la verification des cibles est ignoree.')

print()
orphelins = {}
for sorte, lst in groupes.items():
    for cible in lst:
        present = cible in identifiants_stock
        statut = 'present' if present else 'ABSENT (orphelin)'
        if not present:
            orphelins.setdefault(sorte, []).append(cible)
        print(f'{sorte:10} {cible:18} -> {statut}')

print()
print('=== Recapitulatif des orphelins ===')
for sorte in ['Actualite','Contenu','Fichier','Autre']:
    lst = sorted(set(orphelins.get(sorte, [])))
    if lst:
        print(f'{sorte} orphelins ({len(lst)}) : ' + ', '.join(lst))

Identifiants connus dans le stock : 6311

Actualite  12500-PGP          -> ABSENT (orphelin)
Fichier    13093-PGP          -> ABSENT (orphelin)
Fichier    13694-PGP          -> ABSENT (orphelin)
Fichier    13695-PGP          -> ABSENT (orphelin)
Fichier    13705-PGP          -> ABSENT (orphelin)
Fichier    13706-PGP          -> ABSENT (orphelin)
Fichier    14122-PGP          -> ABSENT (orphelin)
Contenu    13690-PGP          -> ABSENT (orphelin)
Contenu    13691-PGP          -> ABSENT (orphelin)
Contenu    5176-PGP           -> present
Contenu    5251-PGP           -> present
Contenu    5261-PGP           -> present
Contenu    5266-PGP           -> present
Contenu    5312-PGP           -> present
Contenu    8346-PGP           -> present

=== Recapitulatif des orphelins ===
Actualite orphelins (1) : 12500-PGP
Contenu orphelins (2) : 13690-PGP, 13691-PGP
Fichier orphelins (6) : 13093-PGP, 13694-PGP, 13695-PGP, 13705-PGP, 13706-PGP, 14122-PGP


## Etape 6. Ouvrir le texte du document (facultatif)

Enregistre le contenu HTML dans un dossier local, ouvrable d'un double-clic dans un navigateur.

In [6]:
if fichiers_doc and os.path.exists(TGZ_PATH):
    htmls = [n for n in fichiers_doc if n.lower().endswith(('.html','.htm'))]
    if htmls:
        dossier = os.path.join(os.path.dirname(TGZ_PATH), 'extrait_' + IDENTIFIANT)
        os.makedirs(dossier, exist_ok=True)
        with tarfile.open(TGZ_PATH, 'r:gz') as tar:
            data = tar.extractfile(htmls[0]).read()
        chemin = os.path.join(dossier, IDENTIFIANT + '.html')
        with open(chemin, 'wb') as f:
            f.write(data)
        print('Fichier HTML enregistre ici :')
        print('   ', chemin)
    else:
        print('Aucun fichier de contenu .html trouve.')

Fichier HTML enregistre ici :
    ./data/extrait_8347-PGP/8347-PGP.html
